In [36]:
import os
import re

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

from collections import Counter

import joblib

In [37]:
LOG_FILE = r"../../Dataset/HDFS/HDFS_1/HDFS.log"

LABEL_FILE = r"../../Dataset/HDFS/HDFS_1/anomaly_label.csv"

In [38]:
print(os.path.exists(LOG_FILE))

print(os.path.exists(LABEL_FILE))

True
True


In [39]:
labels_df = pd.read_csv(LABEL_FILE)

labels_df.head()

,BlockId,Label
0,blk_-1608999687919862906,Normal
1,blk_7503483334202473044,Normal
2,blk_-3544583377289625738,Anomaly
3,blk_-9073992586687739851,Normal
4,blk_7854771516489510256,Normal


In [40]:
print(labels_df.columns)

Index(['BlockId', 'Label'], dtype='str')


In [41]:
with open(
    LOG_FILE,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:

    logs = f.readlines()

print("Total Logs:", len(logs))

Total Logs: 11175629


In [42]:
block_pattern = r'(blk_-?\d+)'

In [43]:
block_logs = {}

for line in tqdm(logs):

    match = re.search(block_pattern, line)

    if match:

        block_id = match.group(1)

        if block_id not in block_logs:

            block_logs[block_id] = []

        block_logs[block_id].append(line.strip())

100%|██████████| 11175629/11175629 [00:29<00:00, 380404.53it/s]


In [44]:
print("Total Blocks:", len(block_logs))

Total Blocks: 575061


In [45]:
data = []

for block_id, log_lines in block_logs.items():

    combined_logs = " ".join(log_lines)

    data.append({

        "BlockId": block_id,

        "Logs": combined_logs
    })

df = pd.DataFrame(data)

df.head()

,BlockId,Logs
0,blk_-1608999687919862906,081109 203518 143 INFO dfs.DataNode$DataXceive...
1,blk_7503483334202473044,081109 203520 142 INFO dfs.DataNode$DataXceive...
2,blk_-3544583377289625738,081109 203521 145 INFO dfs.DataNode$DataXceive...
3,blk_-9073992586687739851,081109 203523 143 INFO dfs.DataNode$DataXceive...
4,blk_7854771516489510256,081109 203529 148 INFO dfs.DataNode$DataXceive...


In [46]:
# =========================================
# MERGE LOGS WITH LABELS
# =========================================

df = df.merge(

    labels_df,

    on="BlockId",

    how="inner"
)

print(df.shape)

df.head()

(575061, 3)


,BlockId,Logs,Label
0,blk_-1608999687919862906,081109 203518 143 INFO dfs.DataNode$DataXceive...,Normal
1,blk_7503483334202473044,081109 203520 142 INFO dfs.DataNode$DataXceive...,Normal
2,blk_-3544583377289625738,081109 203521 145 INFO dfs.DataNode$DataXceive...,Anomaly
3,blk_-9073992586687739851,081109 203523 143 INFO dfs.DataNode$DataXceive...,Normal
4,blk_7854771516489510256,081109 203529 148 INFO dfs.DataNode$DataXceive...,Normal


In [47]:
print("\nLabel Distribution:\n")

print(df["Label"].value_counts())


Label Distribution:

Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64


In [48]:
label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    df["Label"]
)

In [49]:
for i, label in enumerate(label_encoder.classes_):

    print(i, "->", label)

0 -> Anomaly
1 -> Normal


In [50]:
# =========================================
# SPLIT RAW TEXT FIRST
# =========================================

X_text = df["Logs"]

X_train_text, X_test_text, y_train, y_test = train_test_split(

    X_text,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print("\nTraining Samples:", len(X_train_text))

print("Testing Samples:", len(X_test_text))


Training Samples: 460048
Testing Samples: 115013


In [51]:
# =========================================
# TF-IDF FEATURE ENGINEERING
# =========================================

vectorizer = TfidfVectorizer(

    max_features=5000,

    stop_words='english'
)

# FIT ONLY ON TRAIN DATA

X_train = vectorizer.fit_transform(
    X_train_text
)

# TRANSFORM TEST DATA

X_test = vectorizer.transform(
    X_test_text
)

print("\nTrain Shape:", X_train.shape)

print("Test Shape:", X_test.shape)


Train Shape: (460048, 5000)
Test Shape: (115013, 5000)


In [52]:
print("\nTraining Distribution:\n")

print(Counter(y_train))


Training Distribution:

Counter({np.int64(1): 446578, np.int64(0): 13470})


In [53]:
print("\nAnomaly Ratio:")

print(df["Label"].value_counts(normalize=True))


Anomaly Ratio:
Label
Normal     0.97072
Anomaly    0.02928
Name: proportion, dtype: float64


In [54]:
model = XGBClassifier(

    n_estimators=150,

    max_depth=5,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    objective='binary:logistic',

    eval_metric='logloss',

    n_jobs=-1,

    random_state=42
)

In [55]:
model.fit(

    X_train,
    y_train,

    eval_set=[(X_test, y_test)],

    verbose=True
)

[0]	validation_0-logloss:0.08591
[1]	validation_0-logloss:0.07580
[2]	validation_0-logloss:0.06837
[3]	validation_0-logloss:0.06245
[4]	validation_0-logloss:0.05749
[5]	validation_0-logloss:0.05337
[6]	validation_0-logloss:0.04962
[7]	validation_0-logloss:0.04626
[8]	validation_0-logloss:0.04325
[9]	validation_0-logloss:0.04060
[10]	validation_0-logloss:0.03811
[11]	validation_0-logloss:0.03584
[12]	validation_0-logloss:0.03374
[13]	validation_0-logloss:0.03189
[14]	validation_0-logloss:0.03009
[15]	validation_0-logloss:0.02841
[16]	validation_0-logloss:0.02686
[17]	validation_0-logloss:0.02540
[18]	validation_0-logloss:0.02404
[19]	validation_0-logloss:0.02283
[20]	validation_0-logloss:0.02168
[21]	validation_0-logloss:0.02055
[22]	validation_0-logloss:0.01949
[23]	validation_0-logloss:0.01860
[24]	validation_0-logloss:0.01767
[25]	validation_0-logloss:0.01679
[26]	validation_0-logloss:0.01595
[27]	validation_0-logloss:0.01522
[28]	validation_0-logloss:0.01450
[29]	validation_0-loglos

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [56]:
y_prob = model.predict_proba(X_test)

y_pred = np.argmax(y_prob, axis=1)

In [57]:
train_pred = model.predict(X_train)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

print(f"\nTraining Accuracy: {train_accuracy:.4f}")


Training Accuracy: 0.9999


In [58]:
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.4f}")


Test Accuracy: 0.9998


In [59]:
print(

    classification_report(

        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3368
           1       1.00      1.00      1.00    111645

    accuracy                           1.00    115013
   macro avg       1.00      1.00      1.00    115013
weighted avg       1.00      1.00      1.00    115013



In [60]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[  3355     13]
 [    10 111635]]


In [61]:
# =========================================
# CHECK OVERFITTING GAP
# =========================================

gap = train_accuracy - accuracy

print(f"\nOverfitting Gap: {gap:.4f}")


Overfitting Gap: 0.0001


In [62]:
os.makedirs(

    "../../trained_models/hdfs",

    exist_ok=True
)

joblib.dump(

    model,

    "../../trained_models/hdfs/hdfs_xgboost_model.pkl"
)

['../../trained_models/hdfs/hdfs_xgboost_model.pkl']

In [63]:
joblib.dump(

    vectorizer,

    "../../trained_models/hdfs/hdfs_vectorizer.pkl"
)

['../../trained_models/hdfs/hdfs_vectorizer.pkl']

In [64]:
joblib.dump(

    label_encoder,

    "../../trained_models/hdfs/hdfs_label_encoder.pkl"
)

['../../trained_models/hdfs/hdfs_label_encoder.pkl']

In [65]:
print("===================================")

print("AEGIS HDFS Anomaly Detection Ready")

print("===================================")

print("Distributed Anomaly Model Saved")

AEGIS HDFS Anomaly Detection Ready
Distributed Anomaly Model Saved
